# P7 — Trace Transformer shapes (CPU)

Optional companion, no TODO functions. Run top-to-bottom. Begin with B=2, T=5, C=32, H=4. Change only T=5 to T=3, rerun, then restore 5. Track every shape before looking at numeric values.


In [ ]:
import math
import torch

torch.manual_seed(5)
B, T, C, H = 2, 5, 32, 4  # change T to 3, then restore 5
D = C // H
assert C % H == 0
x = torch.randn(B, T, C)
q = x.reshape(B, T, H, D).transpose(1, 2)
k = q.clone()
v = q.clone()
print("x:", tuple(x.shape), "q:", tuple(q.shape))
assert q.shape == (B, H, T, D)


## Scores: last two dimensions swap

`matmul` combines the head-specific channel dimension D; attention scores end in (T,T). A True upper-triangle cell denotes a future position that must be blocked **before** softmax.


In [ ]:
scores = torch.matmul(q, k.transpose(-2, -1)) / math.sqrt(D)
mask = torch.triu(torch.ones(T, T, dtype=torch.bool), diagonal=1)
masked_scores = scores.masked_fill(mask, float("-inf"))
weights = torch.softmax(masked_scores, dim=-1)
print("scores:", tuple(scores.shape), "weights:", tuple(weights.shape))
assert scores.shape == (B, H, T, T)
assert float(weights.masked_select(mask).abs().sum()) == 0.0
assert torch.allclose(weights.sum(dim=-1), torch.ones(B, H, T))


## Multiply weights by V and join heads

`weights @ v` is (B,H,T,D). `transpose(1,2)` returns the layout (B,T,H,D); `contiguous().view(B,T,C)` merges channels back. `reshape` may copy when required; `view` requires compatible strides, so call `contiguous()` after transpose if using `view`.


In [ ]:
head_out = weights @ v
out = head_out.transpose(1, 2).contiguous().view(B, T, C)
print("head_out:", tuple(head_out.shape), "out:", tuple(out.shape))
assert head_out.shape == (B,H,T,D)
assert out.shape == x.shape
print("PASS: causal masks and tensor shapes")
